In [ ]:
import time
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torchvision
import albumentations as A
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Variable
from torchvision.models import segmentation as seg_models
from torchvision import transforms as T
from torchvision.transforms import functional as F
from torchsummary import summary
from tqdm import tqdm
from typing import List, Dict, Tuple, Union, Final
import sys

sys.path.append('../utils')
from json_parser import CellMaskDataset

## Configurations
### Model parameters

In [ ]:
# number of classes (including background)
# the order of objects when creating the semnatic masks is important for semantic segmentation
# we create the semnatic masks in this order: bg, cage, cell, and then bead
# as cells can be inside cages (creating holes in cage masks), and beads
# can potentially be over the cells (creating holes)
LABEL_MAP: Dict[int, str] = {1: 'cage', 2: 'cell', 3: 'bead'}
NUM_CLASSES: Final[int] = len(LABEL_MAP) + 1

MODEL_PATH = 'checkpoints'
if not os.path.exists(MODEL_PATH):
    os.mkdir(MODEL_PATH)

# model input image large/small-side sizes
MODEL_INPUT_SIZE: Final[int] = 512

### Dataset parameters

In [ ]:
ANNOTATIONS_CLASS_NAMES_TO_CLASS_IDS_MAP: Dict[str, int] = {
    'cage': 1,
    'cages': 1,
    'cell': 2,
    # 'dead-cell': 2, 
    'cytoplasm': 2,
    'cell-adhered': 2, 
    # 'soma': 2,
    'bead': 3,
}

# whether only use the in-focus image to create the cage crops
ONLY_USE_BEST_FOCUS_IMAGE: bool = False
# max image size in the dataset
MAX_IMAGE_SIDE: int = 4512
NUM_RANDOM_CROPS_PER_IMG_FOR_UNCAGED_DATASETS: int = 50
PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES: float = 0.2
    
# the datasets to use for creating the cage crops
# note that we have included uncaged datasets as well
# for those sets, we randomly generate a given number of MODEL_INPUT_MAX_SIZE x MODEL_INPUT_MIN_SIZE crops over the image
DATASET_PATHS: Dict[str, List[str]] = {
    'IMR90':   ['/media/cellareye/SSD/Data/Cellanome/Segmentation/231212_imr90_multichannel_overlay',
                '/media/cellareye/SSD/Data/Cellanome/Segmentation/240213_imr90_multichannel_overlay'],
    'Hs675T':  ['/home/cellareye/Cellanome/Data/20240509_Hs675Tfibroblasts_10x_caged',
                '/home/cellareye/Cellanome/Data/20241003_Hs675Tfibroblasts-suspension-beads_10x_uncaged'], 
    'HeLa':    ['/home/cellareye/Cellanome/Data/20240509_hela-adhered_10x_caged'],
    'MC38':    ['/home/cellareye/Cellanome/Data/20240624_mc38_10x_caged',
                '/home/cellareye/Cellanome/Data/20240624_mc38_10x_uncaged',
                '/home/cellareye/Cellanome/Data/20240625_mc38_10x_caged'],
    'mutuDC':  ['/home/cellareye/Cellanome/Data/20240515_DC-adhered_10x_caged', 
                '/home/cellareye/Cellanome/Data/20240516_DC-adhered_10x_caged'],
    # 'Neuron':  ['/home/cellareye/Cellanome/Data/20240422_neuron-adhered_10x_uncaged',  
    #             '/home/cellareye/Cellanome/Data/20240703_neuron-adhered_10x_caged', 
    #             '/home/cellareye/Cellanome/Data/20240704_neuron-adhered_10x_caged'],
    'Jurkat':  ['/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_jurkat_10x_caged',
                '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_jurkat_10x_uncaged'],
    'K562':    ['/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_k562_10x_caged',
                '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_k562_10x_uncaged'],
    'Raji':    ['/home/cellareye/Cellanome/Data/20240905_raji_10x_caged_at_4x'],
    'HeLa':    ['/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_hela-suspension_10x_caged',
                '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_hela-suspension_10x_uncaged'],
    'NK92':    ['/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_nk92_10x_caged',
                '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_nk92_10x_uncaged',
                '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240425_nk92_10x_caged',
                '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240425_nk92_10x_uncaged'],
    'IMR90':   ['/media/cellareye/SSD/Data/Cellanome/Segmentation/20240314_imr90-suspension_10x_caged',
                '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240314_imr90-suspension_10x_uncaged'],
    'PBMC':    ['/media/cellareye/SSD/Data/Cellanome/Segmentation/20240307_pbmc-beads_10x_uncaged',
                '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240305_pbmc-nobeads_10x_caged',
                '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240305_pbmc-nobeads_10x_uncaged',
                '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240306_mousepbmc-beads_10x_caged',
                '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240306_mousepbmc-beads_10x_uncaged',
                '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240306_mousepbmc-nobeads_10x_caged',
                '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240306_mousepbmc-nobeads_10x_uncaged'],
    'TALL104': ['/home/cellareye/Cellanome/Data/20240816_tall104_10x_caged_at_4x'], 
    'U87':     ['/home/cellareye/Cellanome/Data/20240905_u87-adhered_10x_caged'], 
    'Glia':    ['/home/cellareye/Cellanome/Data/20240924_enteric-glia-adhered_10x_uncaged'],
    'iNeuron': ['/home/cellareye/Cellanome/Data/20240813_ineuron-adhered-1day_10x_caged']
}


OUTPUT_FOLDER = 'cage_crops_data'

if not os.path.exists(OUTPUT_FOLDER):
    os.mkdir(OUTPUT_FOLDER)

## Create cropped images of cages
### Skip this step onces the training images and semantic masks are created! 
This step should be repeated for any additional dataset/cell type available.

If the dataset is not caged, we randomly selects a specified number of MODEL_INPUT_MAX_SIZE x MODEL_INPUT_MIN_SIZE crops over the image.

In [ ]:
# a helper function to parse the annotations and return the dataset class provided the path to the dataset
def create_dataset_class(dataset_path: str, 
                         class_names_to_class_ids_map: Dict[str, int] = ANNOTATIONS_CLASS_NAMES_TO_CLASS_IDS_MAP, 
                         only_use_best_focus_image: bool = ONLY_USE_BEST_FOCUS_IMAGE,
                         train: bool = True, 
                         percentage_to_expand_bbox_boundaries: float = PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES, 
                         max_larger_side: int = MAX_IMAGE_SIDE, 
                         max_smaller_side: int = MAX_IMAGE_SIDE):
    
    images_path:str = dataset_path
    annotations_path: str = os.path.join(dataset_path, 'annotations')

    annotations_images_map: pd.DataFrame = pd.read_csv(os.path.join(dataset_path, 'annotation_images_mapping.csv'))

    test_files: List[str] = []
    train_files: List[str] = []
    
    with open(os.path.join(dataset_path, 'test.txt')) as file:
        filenames = file.readlines()
        filenames = [f.replace('\n', '') for f in filenames if len(f) > 0]
        test_files += filenames

    with open(os.path.join(dataset_path, 'train.txt')) as file:
        filenames = file.readlines()
        filenames = [f.replace('\n', '') for f in filenames if len(f) > 0]
        train_files += filenames

 
    columns: List[str] = list(annotations_images_map.columns)
    image_columns = [column_name for column_name in columns if 'white_dz' in column_name.lower()]

    if len(image_columns) > 0 and only_use_best_focus_image:
        image_columns = [column_name for column_name in columns if 'white_dz0' in column_name.lower()]
        
    if len(image_columns) == 0:
        # this is not a focus sweep dataset, get the BF image
        for column_name in columns:
            if 'bf' in column_name.lower() or 'white' in column_name.lower():
                image_columns = [column_name]
                break

    if len(image_columns) == 0:
        print('[ERROR]: No brightfield image folder could be extracted from annotation_images_mapping.csv file for the dataset')
    
    train_map_dict: Dict[str, List[str]] = {}
    test_map_dict: Dict[str, List[str]] = {}
    for _, row in annotations_images_map.iterrows():
        annotations_filename = row['annotation_json']
        name = '.'.join(annotations_filename.strip().split('.')[:-1])
        if name in test_files:
            test_map_dict[annotations_filename] = list(row[image_columns])
        else:
            train_map_dict[annotations_filename] = list(row[image_columns])


    # datasets
    if train:
        dataset = CellMaskDataset(images_path=images_path, annotations_path=annotations_path, 
                                  annotations=train_map_dict,
                                  max_images_to_consider_for_each_annotation=1, # in case of focus sweep dataset, only pick 1 image randomly
                                  labels_of_interest=list(class_names_to_class_ids_map.keys()), 
                                  percentage_to_expand_bbox_boundaries = percentage_to_expand_bbox_boundaries, 
                                  color_depth=8, 
                                  min_object_diameter = 6.0,
                                  scale_factor_dict={}, 
                                  max_larger_side = max_larger_side, max_smaller_side = max_smaller_side,
                                  normalize=False, class_names_to_ids_map=class_names_to_class_ids_map)

    else:
        dataset = CellMaskDataset(images_path=images_path, annotations_path=annotations_path, 
                                  annotations=test_map_dict,
                                  max_images_to_consider_for_each_annotation=1,
                                  labels_of_interest=list(class_names_to_class_ids_map.keys()), 
                                  percentage_to_expand_bbox_boundaries = percentage_to_expand_bbox_boundaries, 
                                  color_depth=8, 
                                  min_object_diameter = 6.0,
                                  scale_factor_dict={}, 
                                  max_larger_side = max_larger_side, max_smaller_side = max_smaller_side,
                                  normalize=False, class_names_to_ids_map=class_names_to_class_ids_map)

    return dataset

In [ ]:
def build_semantic_mask(data_sample: dict, class_ids_of_interest: List[int]):

    img_height, img_width = data_sample['image'].shape[:2]
    
    # semantic mask in full image resolution
    # we are using np.uint8, hence only 255 segments (which is fine)    
    semantic_mask: np.ndarray = np.zeros((img_height, img_width), np.uint8) # zero is the background always

    for class_id in class_ids_of_interest:
        # go over the classes in the provided order to ensure the order of overlapping objects
        # we create separate semantic masks for all the objects of the same class first, and then overlay them
        objs_of_interest_df: pd.DataFrame = data_sample['annotations'][data_sample['annotations']['label'] == class_id]
        semantic_mask_this_class = np.zeros((img_height, img_width), np.uint8)
        for row_idx, row in objs_of_interest_df.iterrows():
            xmin, ymin, xmax, ymax = row[['xtl', 'ytl', 'xbr', 'ybr']]
            # no need to check the validity 
            if xmin >= xmax or ymin >= ymax:
                continue
                
            # update the semantic mask
            instance_mask: np.ndarray = data_sample['masks'][row_idx] * class_id 
            # only update the non-zero areas, otherwise, we may remove parts of
            # the semantic mask from other instances (note that these are all from the same class)
            semantic_mask_this_class[ymin:ymax, xmin:xmax][instance_mask > 0] = instance_mask[instance_mask > 0]
        # now combine the semantic masks for different classes
        # since we are going over the class IDs in a given order, cage masks will be replaced by cells, and 
        # cell masks will be replaced by beads
        semantic_mask = np.maximum(semantic_mask, semantic_mask_this_class)
    return semantic_mask


def generate_random_crops(img: np.ndarray, 
                          semantic_mask: np.ndarray, 
                          crop_size: int, 
                          scale_range: Tuple[float, float], 
                          num_crops: int
                         ):
    
    img_height, img_width = img.shape[:2]
    crops_top_left_x: np.ndarray = (max(0, img_width - crop_size) * np.random.rand(num_crops)).astype(int)
    crops_top_left_y: np.ndarray = (max(0, img_height - crop_size) * np.random.rand(num_crops)).astype(int)
    
    if scale_range[0] > scale_range[1]:
        # invalid scale, do not scale anything
        scale_min: float = 1.0
        scale_range: float = 0.0
    else:
        scale_min: float = scale_range[0]
        scale_range: float = scale_range[1] - scale_range[0]
        
    scale: np.ndarray = scale_range * np.random.rand(num_crops) + scale_min

    img_crops: List[np.ndarray] = []
    mask_crops: List[np.ndarray] = []
    for i in range(num_crops):
        xc_tl: int = crops_top_left_x[i]
        yc_tl: int = crops_top_left_y[i]
        xc_br: int = crops_top_left_x[i] + int(crop_size * scale[i])
        yc_br: int = crops_top_left_y[i] + int(crop_size * scale[i])

        if xc_br > img_width:
            xc_br = img_width
            xc_tl = max(0, xc_br - int(crop_size * scale[i]))

        if yc_br > img_height:
            yc_br = img_height
            yc_tl = max(0, yc_br - int(crop_size * scale[i]))

        # resize the random crop to the original size
        if scale[i] > 1:
            interpolation_scheme = cv2.INTER_AREA
        else:
            interpolation_scheme = cv2.INTER_CUBIC

        # skip the crops that only include background 
        unique_labels: np.ndarray = np.unique(semantic_mask[yc_tl:yc_br, xc_tl:xc_br])
        
        if len(unique_labels) == 1 and unique_labels[0] == 0:
            continue
        
        cropped_img: np.ndarray = cv2.resize(img[yc_tl:yc_br, xc_tl:xc_br], 
                                             dsize=(crop_size, crop_size), 
                                             interpolation=interpolation_scheme)
        cropped_mask: np.ndarray = cv2.resize(semantic_mask[yc_tl:yc_br, xc_tl:xc_br], 
                                              dsize=(crop_size, crop_size), 
                                              interpolation=cv2.INTER_NEAREST)
        
        
        
        img_crops.append(cropped_img)
        mask_crops.append(cropped_mask)

    return img_crops, mask_crops

def generate_cage_crops(img: np.ndarray, 
                        semantic_mask: np.ndarray, 
                        in_cage_boxes: np.ndarray, 
                        crop_size: int, 
                       ):
    
    # image height and width
    img_height, img_width = img.shape[:2]
    
    # outputs
    img_crops: List[np.ndarray] = []
    mask_crops: List[np.ndarray] = []

    half_crop_size: int = int(np.ceil(crop_size / 2.0))
    
    # make a copy to not modify the input
    cage_boxes: np.ndarray = in_cage_boxes.copy()
    
    cage_boxes_widths: np.ndarray = cage_boxes[:, 2] - cage_boxes[:, 0]
    cage_boxes_heights: np.ndarray = cage_boxes[:, 3] - cage_boxes[:, 1]
    
    # the center of cages (defined as the center of cage boxes)
    cage_centers_x: np.ndarray = (cage_boxes[:, 2] + cage_boxes[:, 0]) / 2.0
    cage_centers_y: np.ndarray = (cage_boxes[:, 3] + cage_boxes[:, 1]) / 2.0
        
    half_cage_sizes: np.ndarray = np.maximum(cage_boxes_heights, cage_boxes_widths) / 2.0
    
    # expand the bbox to create a square bounding box around the object
    # we do this to avoid having black bands on top/bottom or left/right of the images
    # note that the bounding boxes for objects at the image boundaries may not be sqaure, but we 
    # can ignore those cases for now
    cage_boxes[:, 0] = np.maximum((cage_centers_x - half_cage_sizes), 0).astype(int)
    cage_boxes[:, 1] = np.maximum((cage_centers_y - half_cage_sizes), 0).astype(int)
    cage_boxes[:, 2] = np.minimum((cage_centers_x + half_cage_sizes), img_width).astype(int)
    cage_boxes[:, 3] = np.minimum((cage_centers_y + half_cage_sizes), img_height).astype(int)
        
    for cage_id, (xtl, ytl, xbr, ybr) in enumerate(cage_boxes):
                   
        # form an sqaure crop around the cage 
        # randomly offset the center of this crop around the center of the box (center of the cage)
        # note that the bounding box of the cage is already expanded by the factor PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES
        # when the dataset is created, so we have this much margin around the cage mask and we can offset this much
        # we limit this offset further by a factor of 0.75
        half_cage_size: int = int(half_cage_sizes[cage_id])
        crop_center_offset_in_x: float = (np.random.rand() - 0.5) * PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES * cage_boxes_widths[cage_id] * 0.75
        start_pixel_in_x: int = int(max(0, cage_centers_x[cage_id] - half_cage_size + crop_center_offset_in_x))
        end_pixel_in_x: int = start_pixel_in_x + 2 * half_cage_size
        # adjust for the objects on the image boundaries
        if end_pixel_in_x >= img_width:
            end_pixel_in_x = img_width
            start_pixel_in_x = img_width - 2 * half_cage_size
        crop_center_offset_in_y: float = (np.random.rand() - 0.5) * PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES * cage_boxes_heights[cage_id] * 0.75
        start_pixel_in_y: int = int(max(0, cage_centers_y[cage_id] - half_cage_size + crop_center_offset_in_y))
        end_pixel_in_y: int = start_pixel_in_y + 2 * half_cage_size
        if end_pixel_in_y >= img_height:
            end_pixel_in_y = img_height
            start_pixel_in_y = img_height - 2 * half_cage_size
        
        # resize the cage crop within the provided crop size
        if half_cage_size > half_crop_size:
            interpolation_scheme = cv2.INTER_AREA
        else:
            interpolation_scheme = cv2.INTER_CUBIC
        cropped_img: np.ndarray = cv2.resize(img[start_pixel_in_y:end_pixel_in_y, start_pixel_in_x:end_pixel_in_x], 
                                             dsize=(crop_size, crop_size), 
                                             interpolation=interpolation_scheme)
        cropped_mask: np.ndarray = cv2.resize(semantic_mask[start_pixel_in_y:end_pixel_in_y, start_pixel_in_x:end_pixel_in_x], 
                                             dsize=(crop_size, crop_size), 
                                             interpolation=cv2.INTER_NEAREST)
        img_crops.append(cropped_img)
        mask_crops.append(cropped_mask)
    
    return img_crops, mask_crops
                                              
def create_crops(dataset_id: int, 
                 dataset: CellMaskDataset, 
                 sem_seg_label_map: Dict[int, str],
                 output_base_folder: str, 
                 crop_size: int, 
                 train: bool,
                ):

    reverse_label_map = {v: k for k, v in sem_seg_label_map.items()}

    # create output folders for cropped images and masks
    if not os.path.exists(os.path.join(output_base_folder, "images")):
        os.mkdir(os.path.join(output_base_folder, "images"))

    if not os.path.exists(os.path.join(output_base_folder, "masks")):
        os.mkdir(os.path.join(output_base_folder, "masks"))

    set_type_str: str = "train" if train else "test"
    # create output folders depending on train/test set
    output_images_folder: str = os.path.join(output_base_folder, "images", set_type_str)
    output_masks_folder: str = os.path.join(output_base_folder, "masks", set_type_str)
    
    if not os.path.exists(output_images_folder):
        os.mkdir(output_images_folder)

    if not os.path.exists(output_masks_folder):
        os.mkdir(output_masks_folder)

    # go over all images in the dataset and generate the crops
    for idx in range(len(dataset)):
        # read the sample 
        sample: dict = dataset[idx] 
        if len(sample['annotations']) == 0:
            continue
        
        # image
        img: np.ndarray = sample['image']

        img_height, img_width = img.shape[:2]
        
        img_name: str = '.'.join(sample['name'].strip().split('.')[:-1])
       
        # first, generate the semantic mask for the whole image before cropping  
        # semantic mask in full image resolution
        semantic_mask: np.ndarray = build_semantic_mask(data_sample=sample, class_ids_of_interest=list(sem_seg_label_map.keys()))

        # now identify the cages
        cages_df: pd.DataFrame = sample['annotations'][sample['annotations']['label'] == reverse_label_map['cage']]

        if len(cages_df) == 0:
            # there is no cages, randomly select the crops over the image
            img_crops, mask_crops = generate_random_crops(img, 
                                                          semantic_mask, 
                                                          crop_size, 
                                                          (0.5, 1.0), 
                                                          NUM_RANDOM_CROPS_PER_IMG_FOR_UNCAGED_DATASETS)
        else:
            cage_boxes: np.ndarray = cages_df[['xtl', 'ytl', 'xbr', 'ybr']].values.astype(int)
            img_crops, mask_crops = generate_cage_crops(img, 
                                                        semantic_mask, 
                                                        cages_df[['xtl', 'ytl', 'xbr', 'ybr']].values.astype(int), 
                                                        crop_size)
        # sve them to disk
        for crop_count in range(len(img_crops)):                
            cv2.imwrite(os.path.join(output_images_folder, str(dataset_id) + '_' + img_name + '_' + str(crop_count) + '.jpg'), 
                        img_crops[crop_count], [int(cv2.IMWRITE_JPEG_QUALITY), 100])

            cv2.imwrite(os.path.join(output_masks_folder, str(dataset_id) + '_' + img_name + '_' + str(crop_count) + '.png'), 
                        mask_crops[crop_count])
        

In [ ]:
dataset_paths_list: List[str] = []
for cell_type, dataset_paths in DATASET_PATHS.items():
    dataset_paths_list += dataset_paths

for dataset_id, dataset_path in enumerate(dataset_paths_list):

    dataset_name = os.path.basename(dataset_path)
    
    train_dataset = create_dataset_class(dataset_path=dataset_path, 
                                           train=True)
    test_dataset = create_dataset_class(dataset_path=dataset_path, 
                                           train=False)
    
    create_crops(
        dataset_id=dataset_id, 
        dataset=train_dataset, 
        sem_seg_label_map=LABEL_MAP,
        output_base_folder=OUTPUT_FOLDER, 
        crop_size=MODEL_INPUT_SIZE, 
        train=True
    )

    create_crops(
        dataset_id=dataset_id, 
        dataset=test_dataset, 
        sem_seg_label_map=LABEL_MAP,
        output_base_folder=OUTPUT_FOLDER, 
        crop_size=MODEL_INPUT_SIZE, 
        train=False
    )

## Data Model
### Dataset class

In [ ]:
class SemanticMaskDataset(Dataset):
    
    def __init__(self, 
                 images_path: str, 
                 masks_path: str,
                 mean: List[float] = [0.449], 
                 std: List[float] = [0.226],
                 transform = None, 
                 patch_size: int = -1):

        self.images_path: str = images_path
        self.masks_path: str = masks_path
    
        self.img_names: List[str] = os.listdir(images_path)
        self.img_names: List[str] = sorted([f for f in self.img_names if f.strip().split('.')[-1] == 'jpg'])
        
        self.mask_names: List[str] = os.listdir(masks_path)
        self.mask_names: List[str] = sorted([f for f in self.mask_names if f.strip().split('.')[-1] == 'png'])

        img_file_names: List[str] = [".".join(f.strip().split('.')[:-1]) for f in self.img_names]
        mask_file_names: List[str] = [".".join(f.strip().split('.')[:-1]) for f in self.mask_names]

        self.imgs_paths = [os.path.join(images_path, f) for f in self.img_names]
        self.masks_paths = [os.path.join(masks_path, f) for f in self.mask_names]

        self.transform = transform
        self.patch_size: int = patch_size
        self.mean: List[float] = mean
        self.std: List[float] = std
        
        if (len(self.mask_names) != len(self.img_names)):
            print("[ERROR] The mask filenames and the image filenames are not consistent! Dataset will not be instantiated correctly!")
            return 

        if any([img_file_names[i] != mask_file_names[i] for i in range(min(len(img_file_names), len(mask_file_names)))]):
            print("[ERROR] The mask filenames and the image filenames are not consistent! Dataset will not be instantiated correctly!")
         
        
    
    def __len__(self) -> int:
        return len(self.imgs_paths)
    
    def __getitem__(self, idx: int) -> (torch.Tensor, torch.Tensor):
        
        image: np.ndarray = cv2.imread(self.imgs_paths[idx], cv2.IMREAD_UNCHANGED)
        if len(image.shape) < 3:
            # the model expects a 3 channel image, 
            image = np.repeat(np.expand_dims(image, axis=2), 3, axis=2)
        
        # semantic mask
        # we are using np.uint8, hence only 255 segments
        mask_name: str = ".".join(self.img_names[idx].strip().split('.')[:-1]) + '.png'
        semantic_mask: np.ndarray = cv2.imread(os.path.join(self.masks_path, mask_name), cv2.IMREAD_UNCHANGED)
        
        if self.transform is not None:
            augmented = self.transform(image=image, mask=semantic_mask)
            image = augmented['image']
            semantic_mask = augmented['mask']
        
        # convert the image (numpy array) to a torch Tensor and normalize it
        convert_normalize_t = T.Compose([T.ToTensor(), T.Normalize(self.mean, self.std)])
        image_tensor: torch.Tensor = convert_normalize_t(image)
        mask_tensor: torch.Tensor = torch.from_numpy(semantic_mask).long()
        
        if self.patch_size > 0:
            image_tensor, mask_tensor = self.tiles(image_tensor, mask_tensor)
        
        return image_tensor, mask_tensor
    
    def tiles(self, img, mask) -> (torch.Tensor, torch.Tensor):

        img_patches = img.unfold(1, self.patch_size, self.patch_size).unfold(2, self.patch_size, self.patch_size) 
        img_patches  = img_patches.contiguous().view(img.shape[0], -1, self.patch_size, self.patch_size) 
        img_patches = img_patches.permute(1, 0, 2, 3)
        
        mask_patches = mask.unfold(0, self.patch_size, self.patch_size).unfold(1, self.patch_size, self.patch_size)
        mask_patches = mask_patches.contiguous().view(-1, self.patch_size, self.patch_size)
        
        return img_patches, mask_patches

### Image and segmentation mask transforms
Here, we use albumentations package that takes both image and annotations to apply the transformations on them. It takes and returns numpy arrays. 

In [ ]:
train_transform = A.Compose([A.HorizontalFlip(), 
                             A.VerticalFlip(), 
                             A.GridDistortion(p=0.2), 
                             A.RandomBrightnessContrast(brightness_limit = (-0.2, 0.2), 
                                                        contrast_limit = (-0.2, 0.2)),
                             A.GaussNoise()])

### Datasets and Dataloaders

In [ ]:
# datasets
train_dataset = SemanticMaskDataset(images_path=os.path.join(OUTPUT_FOLDER, "images", "train"), 
                                    masks_path=os.path.join(OUTPUT_FOLDER, "masks", "train"),
                                    transform=train_transform)

test_dataset = SemanticMaskDataset(images_path=os.path.join(OUTPUT_FOLDER, "images", "test"), 
                                   masks_path=os.path.join(OUTPUT_FOLDER, "masks", "test"))

## Visualization

In [ ]:
from sem_seg_utils import show_sample, to_numpy

In [ ]:
img = show_sample(31, train_dataset)
Image.fromarray(img[:, :, ::-1])

## Model Definition

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
# model = seg_models.deeplabv3_resnet50(weights=seg_models.DeepLabV3_ResNet50_Weights.DEFAULT) # Load net
model = seg_models.fcn_resnet50(weights=seg_models.FCN_ResNet50_Weights.DEFAULT)

# Change final layer to 4 classes 
# note that FCN and DeepLab models have different number of channels for this layer (512 vs 256 for DeepLab)
num_channels: int =  model.classifier[4].state_dict()['weight'].shape[1]
model.classifier[4] = torch.nn.Conv2d(num_channels, NUM_CLASSES, kernel_size=(1, 1), stride=(1, 1)) 

## Training
### Training parameters

In [ ]:
# training batch size
# this should be at least 2 as the DeepLab model Batch norm require at least 2
BATCH_SIZE: Final[int] = 16
# learning rate
LEARNING_RATE: Final[float] = 1e-3
# number of training epochs
NUM_EPOCHS = 6
# learning rate decay steps, a value of 0 means One-cycle LR scheduler should be used
LR_DECAY_STEPS = 0

### Dataloaders

In [ ]:
# drop_last is set to True to avoid passing a data with batch size of 1
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)     

### Optimizer, LR scheduler and loss function

In [ ]:
# construct an optimizer
# Adam optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr = LEARNING_RATE)

print(f"Adam Optimizer is configured for {NUM_EPOCHS} epochs")

print(f"Initial learning rate is set to {LEARNING_RATE}")
if LR_DECAY_STEPS < 1:
    print(f"One-Cyle LR scheduler is configured for {NUM_EPOCHS} with {len(train_loader)} steps per epoch")
    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer, 
                                                       max_lr=LEARNING_RATE, 
                                                       epochs=NUM_EPOCHS,
                                                       steps_per_epoch=len(train_loader))
    
else:
    print(f"Step LR scheduler is configured with {LR_DECAY_STEPS} epochs for each step")
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer,
                                                   step_size=LR_DECAY_STEPS,
                                                   gamma=0.1)

### Performance metrics

In [ ]:
from sem_seg_utils import mIoU, pixel_accuracy

### Training script

In [ ]:
def train(model, 
          train_loader, 
          test_loader, 
          ignore_index_zero, # set to true if the model should not be trained on the background pixels 
          optimizer,         # (e.g., the background is not annotated properly)
          lr_scheduler,
          num_epochs,
          device,
          patch=False):
    
    torch.cuda.empty_cache()
    
    # losses, accuracies and mean IoUs over the training epochs
    train_losses: List[float] = []
    test_losses: List[float] = []
    train_ious: List[float] = []
    train_accs: List[float] = []
    test_ious: List[float] = [] 
    test_accs: List[float] = []
    
    # learning rates used for each step (not each epoch as we may use OneCyle scheduling)
    lrs: List[float] = []
    min_loss: float = np.inf

   
    if ignore_index_zero:
        criterion = nn.CrossEntropyLoss(ignore_index=0)
    else:
        criterion = nn.CrossEntropyLoss()
    
    model = model.to(device)
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        
        since = time.time()
        
        running_loss: float = 0
        iou_score: float = 0
        accuracy: float = 0

        num_valid_train_batches: int = 0
        num_valid_test_batches: int = 0
        
        # training loop
        model.train()
        for i, data in enumerate(tqdm(train_loader)):
            # training phase
            image_tiles, mask_tiles = data
            if patch:
                bs, n_tiles, c, h, w = image_tiles.size()

                image_tiles = image_tiles.view(-1, c, h, w)
                mask_tiles = mask_tiles.view(-1, h, w)
                
            image = image_tiles.to(device) 
            mask = mask_tiles.to(device)

            if ignore_index_zero:
                unique_mask_values_tensor = mask.unique()
                if len(unique_mask_values_tensor) == 1 and unique_mask_values_tensor[0].item() == 0:
                    # the cross entropy loss ignores index 0 and if the mask does not have any non-zero (non-background) pixels, 
                    # the loss will be nan, so we skip this batch
                    # the training dataset is prepared to only contain images with some non-bg pixels, but this step is included 
                    # for sanity
                    # no need to reset the gradients (yet) as we have not taken any forward step
                    continue
                
            
            # forward
            output = model(image)['out']
            loss = criterion(output, mask)
            
            # evaluate metrics
            # mean IoU can only be np.nan if the mask only include background (index 0) pixles and the mIoU function is called
            # to ignore this index (default) 
            # this should never happen, 
            iou_score += mIoU(output, mask, num_classes=NUM_CLASSES, ignore_index_zero=ignore_index_zero)
            accuracy += pixel_accuracy(output, mask)
            num_valid_train_batches += 1
                
            # backward
            loss.backward()
            optimizer.step() # update weight          
            optimizer.zero_grad() # reset gradient
            
            # update the learning rate only after one batch in case of One-Cycle LR scheduler
            lrs.append(lr_scheduler.get_last_lr()[0])
            if isinstance(lr_scheduler, torch.optim.lr_scheduler.OneCycleLR):
                lr_scheduler.step() 

            # similarly, loss will never be nan
            running_loss += loss.item()
        
        # update the learning rate after one full epoch if LR step scheduler is used
        if isinstance(lr_scheduler, torch.optim.lr_scheduler.StepLR):
            lr_scheduler.step() 
        
        # run the validation after each training epoch
        model.eval()
        test_running_loss: float = 0
        test_iou_score: float = 0
        test_accuracy: float = 0
        
        # validation loop
        with torch.no_grad():
            for i, data in enumerate(tqdm(test_loader)):  
                # reshape to 9 patches from single image, delete batch size
                image_tiles, mask_tiles = data

                if patch:
                    bs, n_tiles, c, h, w = image_tiles.size()
                    image_tiles = image_tiles.view(-1, c, h, w)
                    mask_tiles = mask_tiles.view(-1, h, w)
                    
                image = image_tiles.to(device) 
                mask = mask_tiles.to(device)

                if ignore_index_zero:
                    unique_mask_values_tensor = mask.unique()
                    if len(unique_mask_values_tensor) == 1 and unique_mask_values_tensor[0].item() == 0:
                        continue
                
                output = model(image)['out']
                
                # evaluation metrics
                test_iou_score += mIoU(output, mask, num_classes=NUM_CLASSES, ignore_index_zero=ignore_index_zero)
                test_accuracy += pixel_accuracy(output, mask)
                num_valid_test_batches += 1
                
                # loss 
                loss = criterion(output, mask)                                  
                test_running_loss += loss.item()
            
        # calculatio mean for each batch
        running_loss /= num_valid_train_batches
        iou_score /= num_valid_train_batches
        accuracy /= num_valid_train_batches
        
        test_running_loss /= num_valid_test_batches
        test_iou_score /= num_valid_test_batches
        test_accuracy /= num_valid_test_batches
                       
         # save the results
        train_losses.append(running_loss)
        train_ious.append(iou_score)
        train_accs.append(accuracy)
        
        test_losses.append(test_running_loss)
        test_ious.append(test_iou_score)
        test_accs.append(test_accuracy)
        
        print('saving the model ...')
        torch.save(model.state_dict(), os.path.join(MODEL_PATH, 'checkpoint_' + str(epoch) +'.pt'))
                    
        
        print("Epoch:{}/{} ... \n".format(epoch + 1, num_epochs),
              "Train Loss: {:.3f} \n".format(running_loss),
              "Test Loss: {:.3f} \n".format(test_running_loss),
              "Train mean IoU: {:.3f} \n".format(iou_score),
              "Test mean IoU: {:.3f} \n".format(test_iou_score),
              "Train Accuracy: {:.3f} \n".format(accuracy),
              "Test Accuracy: {:.3f} \n".format(test_accuracy),
              "Time: {:.2f} m".format((time.time() - since) / 60))
        
    history = {'train_loss' : train_losses, 'test_loss': test_losses,
               'train_mean_iou' :train_ious, 'test_mean_iou': test_ious,
               'train_acc': train_accs, 'val_acc': test_accs,
               'lrs': lrs}
    print('Total time: {:.2f} m' .format((time.time()- start_time) / 60))
    return history

In [ ]:
history  = train(model, 
                 train_loader, 
                 test_loader, 
                 False, # do not ignore the bg index 
                 optimizer, 
                 lr_scheduler,
                 NUM_EPOCHS,
                 device,
                 patch=False)

### Saving the best/final model with some model configurations

In [ ]:
BEST_CHECKPOINT = 'checkpoint_5.pt'
model = seg_models.fcn_resnet50(weights=seg_models.FCN_ResNet50_Weights.DEFAULT)
# Change final layer to 2 classes 
# note that FCN and DeepLab models have different number of channels for this layer (512 vs 256 for DeepLab)
num_channels: int =  model.classifier[4].state_dict()['weight'].shape[1]
model.classifier[4] = torch.nn.Conv2d(num_channels, NUM_CLASSES, kernel_size=(1, 1), stride=(1, 1)) 

model.load_state_dict(torch.load(os.path.join(MODEL_PATH, BEST_CHECKPOINT)))
model_param_dict = {}
model_param_dict['model_state_dict'] = model.state_dict()
model_param_dict['label_map'] = LABEL_MAP
model_param_dict['input_size'] = MODEL_INPUT_SIZE
torch.save(model_param_dict, os.path.join(MODEL_PATH, 'final.pt'))

## Testing

In [ ]:
NUM_CLASSES: Final[int] = 4
LABEL_MAP: Dict[int, str] = {1: 'cage', 2: 'cell', 3: 'bead'}

MODEL_PATH = 'checkpoints'
CHECKPOINT_NAME = 'checkpoint_5.pt'
# model input image large/small-side sizes
MODEL_INPUT_SIZE: Final[int] = 512


device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
# model = seg_models.deeplabv3_resnet50(weights=seg_models.DeepLabV3_ResNet50_Weights.DEFAULT)
model = seg_models.fcn_resnet50(weights=seg_models.FCN_ResNet50_Weights.DEFAULT)
# Change final layer to 2 classes 
# note that FCN and DeepLab models have different number of channels for this layer (512 vs 256 for DeepLab)
num_channels: int =  model.classifier[4].state_dict()['weight'].shape[1]
model.classifier[4] = torch.nn.Conv2d(num_channels, NUM_CLASSES, kernel_size=(1, 1), stride=(1, 1)) 

model.load_state_dict(torch.load(os.path.join(MODEL_PATH, CHECKPOINT_NAME)))

In [ ]:
def evaluate(model, data_loader, device, return_loss=False, ignore_index_zero=True):
    # run the validation after each training epoch
    model.eval()
    model.to(device)
    test_iou_score: float = 0
    test_accuracy: float = 0
    test_loss: float = 0
    num_valid_test_batches: int = 0

    if return_loss:
        if ignore_index_zero:
            criterion = nn.CrossEntropyLoss(ignore_index=0)
        else:
            criterion = nn.CrossEntropyLoss()
    
    # validation loop
    with torch.no_grad():
        for i, data in enumerate(tqdm(data_loader)):
            # reshape to 9 patches from single image, delete batch size
            image, mask = data        
            image = image.to(device) 
            mask = mask.to(device)

            unique_mask_values_tensor = mask.unique()
            if ignore_index_zero and len(unique_mask_values_tensor) == 1 and unique_mask_values_tensor[0].item() == 0:
                continue
            
            output = model(image)['out']
            
            # evaluation metrics
            test_iou_score += mIoU(output, mask, num_classes=NUM_CLASSES, ignore_index_zero=ignore_index_zero)
            test_accuracy += pixel_accuracy(output, mask)
            num_valid_test_batches += 1

            if return_loss:
                test_loss += criterion(output, mask).item() 
            
    
    test_iou_score /= num_valid_test_batches
    test_accuracy /= num_valid_test_batches
    if return_loss:
        test_loss /= num_valid_test_batches
        print("Test loss: {:.3f}".format(test_loss))
    
    print("Test mean IoU: {:.3f}".format(test_iou_score))
    print("Test Accuracy: {:.3f}".format(test_accuracy))

In [ ]:
# model.load_state_dict(torch.load(os.path.join(MODEL_PATH, '20240123_sets_1_2_3_6_to_41_0p2_bbox_0p2_b_c_adj_16_bs_10_epochs_1cl_lrs.pt'))['model_state_dict'])
evaluate(model, test_loader, device, return_loss=True, ignore_index_zero=False)

In [ ]:
#  Old model trained ignoring index 0
# Test loss: 0.146 
# Test mean IoU: 0.880 
# Test Accuracy: 0.987 

#  Old model trained including index 0
# Test loss: 0.032
# Test mean IoU: 0.921
# Test Accuracy: 0.987

# Newly trained ignoring index 0
# Test loss: 0.122
# Test mean IoU: 0.880
# Test Accuracy: 0.989

# Newly trained including index 0
# Test loss: 0.026
# Test mean IoU: 0.922
# Test Accuracy: 0.989

In [ ]:
def predict(model, image, device):
    
    mean: Final[float] = 0.449 
    std: Final[float] = 0.226 
    # make sure the image is a gray scale image (BGR or RGB does not matter)
    # then normalize
    if len(image.shape) > 2:
        img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        img = (image / 255.0 - mean) / std
    else:
        img = (image / 255.0 - mean) / std
        
    # convert to tensor and add batch dimension
    img = np.repeat(np.expand_dims(img, axis=2), 3, axis=2)
    
    image_tensor = F.to_tensor(img).unsqueeze(dim=0).to(device).float()
    
    model.eval()
    model.to(device)
    
    with torch.no_grad():
        output = model(image_tensor)["out"]
        mask = torch.argmax(output, dim=1).squeeze().cpu().numpy()
    return mask.astype(np.uint8)

In [ ]:
# idx = 29840
idx = 5000
img_t, mask_t = test_dataset[idx]
image: np.ndarray = to_numpy(img_t.permute(1, 2, 0).squeeze())
mask_gt: np.ndarray = to_numpy(mask_t).astype(np.uint8)
# scale back and add the mean, scale to 0-255
image = ((image * train_dataset.std + train_dataset.mean) * 255).mean(axis=2).astype(np.uint8)
mask = predict(model, image, device)

In [ ]:
Image.fromarray(show_sample(idx, train=False))

In [ ]:
Image.fromarray(mask * 63)

In [ ]:
model.eval()
model.cuda()
output = model(img_t.unsqueeze(0).cuda())['out']
mean_iou = mIoU(output, mask_t.unsqueeze(0).cuda(), ignore_index_zero=True)
accuracy = pixel_accuracy(output, mask_t.unsqueeze(0).cuda())
loss = criterion(output, mask_t.unsqueeze(0).cuda())

In [ ]:
accuracy

In [ ]:
mean_iou

In [ ]:
loss.item()